# GPU Fault Classifier

A PyTorch-based classifier for GPU/node incidents. This notebook:
1. Checks for GPU availability
2. Installs necessary drivers if needed
3. Generates synthetic training data
4. Trains a neural network classifier
5. Evaluates the model and makes predictions

**Features:** `temperature`, `memory_used_pct`, `ecc_errors`, `xid_errors`, `pod_ready`, `driver_ok`, `cuda_visible`, `node_ready`

**Labels:** `normal`, `driver_issue`, `hardware_issue`, `memory_pressure`, `kubernetes_issue`, `node_issue`

In [ ]:
# Step 1: Check for GPU and install drivers
import subprocess
import sys

def check_gpu():
    """Check if GPU is available and print GPU info"""
    print("=" * 60)
    print("GPU Availability Check")
    print("=" * 60)
    
    # Check using nvidia-smi
    try:
        result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
        if result.returncode == 0:
            print("✓ NVIDIA GPU detected!")
            print(result.stdout)
            return True
        else:
            print("✗ nvidia-smi failed")
            print(result.stderr)
            return False
    except FileNotFoundError:
        print("✗ nvidia-smi not found")
        return False

def install_gpu_drivers():
    """Install NVIDIA drivers and CUDA toolkit if needed"""
    print("\n" + "=" * 60)
    print("Installing GPU Drivers")
    print("=" * 60)
    
    # For Google Colab, GPU drivers are usually pre-installed
    # This checks and installs if needed
    
    # Install NVIDIA drivers
    print("Installing nvidia-driver... (if not already installed)")
    subprocess.run([sys.executable, "-m", "pip", "install", "nvidia-driver", "-q"], 
                   capture_output=True)
    
    # Install CUDA toolkit
    print("Installing CUDA toolkit...")
    subprocess.run([sys.executable, "-m", "pip", "install", "cuda-python", "-q"],
                   capture_output=True)
    
    print("Driver installation complete!")

# Check GPU availability
gpu_available = check_gpu()

if gpu_available:
    print("\n✅ GPU is ready for deep learning tasks!")
    install_gpu_drivers()
else:
    print("\n⚠️ No GPU detected. The classifier will run on CPU (slower).")
    print("    For GPU acceleration, ensure you have:")
    print("    - NVIDIA GPU with drivers installed")
    print("    - CUDA toolkit 11.8+ installed")
    print("    - In Google Colab: Runtime > Change runtime type > GPU")

In [ ]:
# Step 2: Install required packages
import subprocess
import sys

print("=" * 60)
print("Installing Required Packages")
print("=" * 60)

packages = [
    "torch>=2.2",
    "scikit-learn>=1.4",
    "numpy>=1.26",
    "fastapi>=0.110",
    "uvicorn[standard]>=0.29",
    "pydantic>=2.0",
    "gradio>=4.0",
]

for package in packages:
    print(f"Installing {package}...")
    subprocess.run([sys.executable, "-m", "pip", "install", package, "-q"], 
                   capture_output=True)

print("\n✅ All packages installed!")

In [ ]:
# Step 3: Set up project structure and generate data
import os
import json
import random
from pathlib import Path

print("=" * 60)
print("Generating Synthetic Training Data")
print("=" * 60)

# Labels for classification
LABELS = ["normal", "driver_issue", "hardware_issue", "memory_pressure", "kubernetes_issue", "node_issue"]

def sample(label):
    """Generate synthetic GPU fault data based on label"""
    x = {
        "temperature": random.gauss(55, 8),
        "memory_used_pct": random.gauss(45, 15),
        "ecc_errors": 0,
        "xid_errors": 0,
        "pod_ready": 1,
        "driver_ok": 1,
        "cuda_visible": 1,
        "node_ready": 1
    }
    if label == "driver_issue":
        x.update(driver_ok=0, cuda_visible=0)
    elif label == "hardware_issue":
        x.update(temperature=random.gauss(88, 5), ecc_errors=random.randint(2, 20), xid_errors=random.randint(1, 8))
    elif label == "memory_pressure":
        x.update(memory_used_pct=random.gauss(94, 3))
    elif label == "kubernetes_issue":
        x.update(pod_ready=0, cuda_visible=0)
    elif label == "node_issue":
        x.update(node_ready=0, pod_ready=0)
    return {"features": x, "label": label}

# Generate data
random.seed(42)
rows = [sample(label) for label in LABELS for _ in range(300)]
random.shuffle(rows)

# Create directories
Path("data").mkdir(exist_ok=True)
Path("artifacts").mkdir(exist_ok=True)

# Save to JSONL
Path("data/gpu_faults.jsonl").write_text("\n".join(json.dumps(r) for r in rows) + "\n")

print(f"✅ Generated {len(rows)} synthetic GPU fault records")
print(f"   Saved to: data/gpu_faults.jsonl")

# Show label distribution
from collections import Counter
label_counts = Counter(r["label"] for r in rows)
print("\nLabel distribution:")
for label, count in label_counts.items():
    print(f"  - {label}: {count} samples")

In [ ]:
# Step 4: Train the GPU Fault Classifier
import json
import numpy as np
import torch
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score, f1_score
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

print("=" * 60)
print("Training GPU Fault Classifier")
print("=" * 60)

# Check device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nUsing device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Define features and labels
FEATURES = ["temperature", "memory_used_pct", "ecc_errors", "xid_errors", 
            "pod_ready", "driver_ok", "cuda_visible", "node_ready"]
LABELS = ["normal", "driver_issue", "hardware_issue", "memory_pressure", 
          "kubernetes_issue", "node_issue"]

# Load data
print("\nLoading training data...")
rows = [json.loads(x) for x in Path("data/gpu_faults.jsonl").read_text().splitlines()]
X = np.array([[r["features"][f] for f in FEATURES] for r in rows], dtype="float32")
y = np.array([LABELS.index(r["label"]) for r in rows])
print(f"Total samples: {len(X)}")

# Split data
Xtr, Xtmp, ytr, ytmp = train_test_split(X, y, test_size=.3, stratify=y, random_state=42)
Xv, Xte, yv, yte = train_test_split(Xtmp, ytmp, test_size=.5, stratify=ytmp, random_state=42)
print(f"Train: {len(Xtr)}, Validation: {len(Xv)}, Test: {len(Xte)}")

# Scale features
scaler = StandardScaler().fit(Xtr)
Xtr, Xv, Xte = [scaler.transform(x).astype("float32") for x in (Xtr, Xv, Xte)]

# Define model
class Classifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(8, 32),
            nn.ReLU(),
            nn.Dropout(.15),
            nn.Linear(32, 6)
        )
    def forward(self, x):
        return self.net(x)

# Initialize model
model = Classifier().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=.003, weight_decay=1e-4)
loss_fn = nn.CrossEntropyLoss()
loader = DataLoader(TensorDataset(torch.tensor(Xtr), torch.tensor(ytr)), 
                    batch_size=64, shuffle=True)

# Training loop
print("\nTraining model...")
for epoch in range(60):
    model.train()
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss_fn(model(xb), yb).backward()
        optimizer.step()
    
    if (epoch + 1) % 10 == 0:
        model.eval()
        with torch.no_grad():
            val_preds = model(torch.tensor(Xv).to(device)).argmax(1).cpu().numpy()
        val_acc = accuracy_score(yv, val_preds)
        print(f"  Epoch {epoch+1}/60 - Validation Accuracy: {val_acc:.4f}")

# Evaluation
print("\n" + "=" * 60)
print("Model Evaluation on Test Set")
print("=" * 60)
model.eval()
with torch.no_grad():
    Xte_tensor = torch.tensor(Xte).to(device)
    predictions = model(Xte_tensor).argmax(1).cpu().numpy()

print(f"\nAccuracy: {accuracy_score(yte, predictions):.4f}")
print(f"Macro F1 Score: {f1_score(yte, predictions, average='macro'):.4f}")

print("\nClassification Report:")
print(classification_report(yte, predictions, target_names=LABELS))

# Save model
torch.save({
    "state_dict": model.state_dict(),
    "mean": scaler.mean_,
    "scale": scaler.scale_,
    "features": FEATURES,
    "labels": LABELS
}, "artifacts/model.pt")
print("\n✅ Model saved to artifacts/model.pt")

In [ ]:
# Step 5: Make Predictions on New Data
import torch
import numpy as np
import torch.nn as nn

# Define Classifier inline (train.py not available in Colab)
class Classifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(8, 32),
            nn.ReLU(),
            nn.Dropout(.15),
            nn.Linear(32, 6)
        )
    def forward(self, x):
        return self.net(x)

print("=" * 60)
print("Making Predictions")
print("=" * 60)

# Load trained model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bundle = torch.load("artifacts/model.pt", map_location=device, weights_only=False)
model = Classifier()
model.load_state_dict(bundle["state_dict"])
model.to(device)
model.eval()

def predict(observation):
    """Predict GPU fault label from observation"""
    features = bundle["features"]
    x = np.array([[observation.get(f, 0) for f in features]], dtype="float32")
    x = ((x - bundle["mean"]) / bundle["scale"]).astype("float32")
    
    with torch.no_grad():
        probs = torch.softmax(model(torch.tensor(x).to(device)), 1)[0].cpu().numpy()
    
    order = probs.argsort()[::-1]
    return {
        "label": bundle["labels"][int(order[0])],
        "confidence": float(probs[order[0]]),
        "probabilities": {bundle["labels"][int(i)]: float(probs[i]) for i in order}
    }

# Example observations to test
test_observations = [
    {
        "temperature": 52,
        "memory_used_pct": 42,
        "ecc_errors": 0,
        "xid_errors": 0,
        "pod_ready": 1,
        "driver_ok": 1,
        "cuda_visible": 1,
        "node_ready": 1
    },  # Normal
    {
        "temperature": 85,
        "memory_used_pct": 45,
        "ecc_errors": 15,
        "xid_errors": 5,
        "pod_ready": 1,
        "driver_ok": 1,
        "cuda_visible": 1,
        "node_ready": 1
    },  # Hardware issue
    {
        "temperature": 58,
        "memory_used_pct": 95,
        "ecc_errors": 0,
        "xid_errors": 0,
        "pod_ready": 1,
        "driver_ok": 1,
        "cuda_visible": 1,
        "node_ready": 1
    },  # Memory pressure
    {
        "temperature": 55,
        "memory_used_pct": 40,
        "ecc_errors": 0,
        "xid_errors": 0,
        "pod_ready": 0,
        "driver_ok": 0,
        "cuda_visible": 0,
        "node_ready": 1
    },  # Driver issue
]

print("\nTest Predictions:")
for i, obs in enumerate(test_observations):
    result = predict(obs)
    print(f"\n[Sample {i+1}] Observed values:")
    print(f"  Temperature: {obs['temperature']}°C")
    print(f"  Memory: {obs['memory_used_pct']}%")
    print(f"  ECC Errors: {obs['ecc_errors']}")
    print(f"  XID Errors: {obs['xid_errors']}")
    print(f"  → Predicted: {result['label']} (confidence: {result['confidence']:.2%})")

In [ ]:
# Step 6: Interactive Gradio Interface
# Gradio provides a web UI for making predictions without blocking the notebook

import gradio as gr
import torch
import numpy as np
import torch.nn as nn

# Define Classifier inline
class Classifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(8, 32),
            nn.ReLU(),
            nn.Dropout(.15),
            nn.Linear(32, 6)
        )
    def forward(self, x):
        return self.net(x)

print("=" * 60)
print("Loading Model...")
print("=" * 60)

# Load trained model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bundle = torch.load("artifacts/model.pt", map_location=device, weights_only=False)
model = Classifier()
model.load_state_dict(bundle["state_dict"])
model.to(device)
model.eval()

def predict(temperature, memory_used_pct, ecc_errors, xid_errors, 
            pod_ready, driver_ok, cuda_visible, node_ready):
    """Predict GPU fault label from observation"""
    observation = {
        "temperature": temperature,
        "memory_used_pct": memory_used_pct,
        "ecc_errors": ecc_errors,
        "xid_errors": xid_errors,
        "pod_ready": 1 if pod_ready else 0,
        "driver_ok": 1 if driver_ok else 0,
        "cuda_visible": 1 if cuda_visible else 0,
        "node_ready": 1 if node_ready else 0
    }
    
    features = bundle["features"]
    x = np.array([[observation.get(f, 0) for f in features]], dtype="float32")
    x = ((x - bundle["mean"]) / bundle["scale"]).astype("float32")
    
    with torch.no_grad():
        probs = torch.softmax(model(torch.tensor(x).to(device)), 1)[0].cpu().numpy()
    
    order = probs.argsort()[::-1]
    
    # Build result string
    result = f"🎯 Predicted: **{bundle['labels'][int(order[0])]}** (confidence: {probs[order[0]]:.1%})\n\n"
    result += "📊 All Probabilities:\n"
    for i in order:
        label = bundle["labels"][int(i)]
        prob = probs[i]
        bar = "█" * int(prob * 20) + "░" * (20 - int(prob * 20))
        result += f"   {label:20s} {bar} {prob:.1%}\n"
    
    return result

# Build Gradio interface
interface = gr.Interface(
    fn=predict,
    title="🚀 GPU Fault Classifier",
    description="Classify GPU/node incidents into 6 categories: normal, driver_issue, hardware_issue, memory_pressure, kubernetes_issue, node_issue",
    inputs=[
        gr.Slider(minimum=-20, maximum=150, value=55, step=1, label="🌡️ Temperature (°C)"),
        gr.Slider(minimum=0, maximum=100, value=45, step=1, label="💾 Memory Used (%)"),
        gr.Slider(minimum=0, maximum=50, value=0, step=1, label="⚠️ ECC Errors"),
        gr.Slider(minimum=0, maximum=20, value=0, step=1, label="⚠️ XID Errors"),
        gr.Checkbox(label="✅ Pod Ready", value=True),
        gr.Checkbox(label="✅ Driver OK", value=True),
        gr.Checkbox(label="✅ CUDA Visible", value=True),
        gr.Checkbox(label="✅ Node Ready", value=True),
    ],
    outputs=gr.Textbox(label="Prediction Result", lines=10),
    examples=[
        [55, 45, 0, 0, True, True, True, True],      # Normal
        [88, 50, 15, 5, True, True, True, True],      # Hardware issue
        [58, 95, 0, 0, True, True, True, True],       # Memory pressure
        [55, 40, 0, 0, False, False, False, True],   # Driver issue
        [55, 40, 0, 0, False, True, True, False],     # Node issue
        [55, 40, 0, 0, False, True, True, True],      # Kubernetes issue
    ],
    article="""
    ## Input Features Guide
    
    | Feature | Description | Normal Range |
    |---------|-------------|--------------|
    | Temperature | GPU temperature in Celsius | 40-70°C |
    | Memory Used % | GPU memory utilization | 20-60% |
    | ECC Errors | ECC correctable errors | 0 |
    | XID Errors | Xid error count | 0 |
    | Pod Ready | K8s pod ready status | ✓ |
    | Driver OK | NVIDIA driver healthy | ✓ |
    | CUDA Visible | CUDA device visible | ✓ |
    | Node Ready | Node schedulable | ✓ |
    
    **Note:** This model was trained on synthetic data. For production use, replace with real `nvidia-smi` and system logs.
    """
)

print("✅ Model loaded successfully!")
print(f"   Device: {device}")
print("\n" + "=" * 60)
print("Starting Gradio Interface...")
print("=" * 60)

# Launch Gradio (share=True creates a public link for Google Colab)
interface.launch(share=True, inline=False)

# Summary

## What This Notebook Does:

1. **GPU Check** - Detects NVIDIA GPU and displays its information

2. **Driver Installation** - Installs necessary NVIDIA drivers and CUDA toolkit

3. **Data Generation** - Creates 1800 synthetic GPU fault records across 6 fault categories

4. **Model Training** - Trains a neural network classifier with:
   - 8 input features (temperature, memory, ECC/XID errors, pod/driver/cuda/node readiness)
   - 32 hidden units with ReLU activation and dropout
   - 6 output classes

5. **Evaluation** - Reports accuracy and macro F1 score on held-out test set

6. **Predictions** - Makes predictions on new GPU health observations

## For Production Use:

Replace `data/gpu_faults.jsonl` with real labeled observations from:
- `nvidia-smi` (temperature, memory usage)
- ECC error logs
- Xid/UE errors
- Kubernetes Events (pod readiness)
- Driver/CUDA version info
- Node health signals

The model artifact is saved to `artifacts/model.pt` and can be deployed via the FastAPI server.